# VEP-nAChR: Complete Project Documentation

**Variant Effect Predictor for Nicotinic Acetylcholine Receptors**

This notebook is a comprehensive guide to the VEP-nAChR project. It explains everything from the biological background to the code implementation, intended for anyone (mentors, colleagues, future students) who needs to understand what this project does and how.

**Table of Contents:**
1. Background: What is a Variant Effect Predictor?
2. Biology: nAChR Proteins and Mutations
3. The Dataset
4. Feature Engineering (the core of this project)
5. Machine Learning Pipeline
6. Comparison with VEP-ENaC (the reference project)
7. Results (with and without structural features)
8. Evaluation Metrics — What They Mean and What's a "Good" Score
9. How Features Are Converted to Numbers (The Math)
10. How to Run the Code
11. Project File Map
12. Next Steps / Roadmap
13. Appendix A: Amino Acid Transfer Statistics
14. Appendix B: Claude Code Skills

---

## 1. Background: What is a Variant Effect Predictor?

### 1.1 The Problem

Human DNA contains instructions for building proteins. Sometimes, a single letter in the DNA changes (a **mutation** or **variant**), which changes one amino acid in the resulting protein. This is called a **missense mutation**.

For example:
```
Normal protein:  ...M-E-L-K-G-A-T...
                       ^
Mutant protein:  ...M-A-L-K-G-A-T...
                       ^
                  Position 97: Glutamic acid (E) → Alanine (A)
```

This single amino acid change can have different effects:
- **Loss-of-Function (LOF):** The protein stops working or works less. Like breaking a switch so it can't turn on.
- **Gain-of-Function (GOF):** The protein becomes overactive or works differently. Like a switch stuck in the ON position.
- **No net effect:** The protein still works normally despite the change.

### 1.2 Why Predict This?

There are thousands of known mutations in nAChR genes, and new ones are discovered through genetic sequencing all the time. Testing each one in a lab (electrophysiology experiments) is expensive and slow. A computational predictor that can look at a mutation and say "this is likely LOF" or "this is likely GOF" based on the amino acid properties and protein structure would be extremely valuable for:
- **Clinical diagnosis:** A patient has a new mutation — is it pathogenic?
- **Drug development:** Understanding which mutations cause disease helps design targeted treatments.
- **Research prioritization:** Which of 500 unstudied mutations should we test in the lab first?

### 1.3 How a VEP Works (High Level)

```
Input: A mutation (e.g., "CHRNA7 E97A")
  ↓
Step 1: Convert the mutation into numerical features
        (physicochemical properties, structural context, etc.)
  ↓
Step 2: Feed the features into a trained ML model
  ↓
Output: Prediction → "LOF" or "GOF"
```

The hard part is Step 1: figuring out WHICH numerical features capture the information that determines whether a mutation causes LOF or GOF. This is called **feature engineering** and is the core of this project.

---

## 2. Biology: nAChR Proteins and Mutations

### 2.1 What are nAChRs?

**Nicotinic acetylcholine receptors (nAChRs)** are proteins that sit in the cell membrane of neurons and muscle cells. They form a channel (a pore) that allows ions (sodium, potassium, calcium) to flow across the membrane when activated.

**How they work:**
1. The neurotransmitter **acetylcholine (ACh)** binds to the receptor
2. This causes the channel to **open**
3. Ions flow through → electrical signal → muscle contracts or neuron fires
4. ACh unbinds → channel **closes**

They are called "nicotinic" because nicotine (from cigarettes) also activates them.

### 2.2 Protein Structure

Each nAChR is a **pentamer** — it is made of 5 subunit proteins assembled in a ring, with the ion channel pore in the center.

```
        Top view (looking down the pore):

            α ---- β
           / \    / \
          /   \  /   \
         δ    [PORE]   α      ← 5 subunits around a central pore
          \   /  \   /
           \ /    \ /
            β/ε --- γ
```

Each subunit has a characteristic structure:
- **Extracellular domain (ECD):** Sticks out of the cell. Contains the ACh binding site at the interface between subunits.
- **Transmembrane domain (TMD):** 4 alpha-helices (TM1-TM4) that span the cell membrane. TM2 lines the ion pore.
- **Intracellular domain (ICD):** A large loop between TM3 and TM4, inside the cell.

### 2.3 The 17 Human nAChR Genes

Humans have 17 genes encoding nAChR subunits. Different combinations form different receptor subtypes in different tissues:

| Gene | Subunit | Location | Receptor Type |
|------|---------|----------|---------------|
| CHRNA1 | α1 | Neuromuscular junction | Muscle-type (α1)₂β1δε |
| CHRNA2 | α2 | Brain | Neuronal heteromeric |
| CHRNA3 | α3 | Autonomic ganglia | α3β4 |
| CHRNA4 | α4 | Brain (widespread) | α4β2 (most common brain nAChR) |
| CHRNA5 | α5 | Brain | Accessory subunit in α4β2α5 |
| CHRNA6 | α6 | Brain (dopamine neurons) | α6β2β3 |
| CHRNA7 | α7 | Brain, immune cells | α7 homomeric (five α7 subunits) |
| CHRNA9 | α9 | Inner ear hair cells | α9α10 |
| CHRNA10 | α10 | Inner ear hair cells | α9α10 |
| CHRNB1 | β1 | Neuromuscular junction | Muscle-type |
| CHRNB2 | β2 | Brain (widespread) | α4β2 |
| CHRNB3 | β3 | Brain | Accessory subunit |
| CHRNB4 | β4 | Autonomic ganglia | α3β4 |
| CHRND | δ | Neuromuscular junction | Muscle-type |
| CHRNE | ε | Neuromuscular junction (adult) | Muscle-type (replaces γ after birth) |
| CHRNG | γ | Neuromuscular junction (fetal) | Fetal muscle-type |

### 2.4 Diseases Caused by nAChR Mutations

| Disease | Subunits | Effect | Symptoms |
|---------|----------|--------|----------|
| Congenital Myasthenic Syndrome (CMS) | CHRNA1, CHRNB1, CHRND, CHRNE | LOF (usually) | Muscle weakness, fatigue |
| Autosomal Dominant Nocturnal Frontal Lobe Epilepsy (ADNFLE) | CHRNA4, CHRNB2, CHRNA2 | GOF (usually) | Seizures during sleep |
| Nicotine dependence (risk factor) | CHRNA5, CHRNA3, CHRNB4 | Various | Increased addiction susceptibility |
| Multiple Pterygium Syndrome | CHRNG | LOF | Fetal akinesia, joint contractures |

### 2.5 LOF vs GOF — What Determines It?

Whether a mutation causes LOF or GOF depends on:
- **Where** in the protein the mutation is (binding site? pore-lining? intracellular?)
- **What** amino acid change occurs (conservative or radical?)
- **How** the change affects the protein's physical properties (charge, size, hydrophobicity)
- **The 3D structural context** (is the position buried or exposed? in a helix or loop?)

This is exactly what our features try to capture.

---

## 3. The Dataset

### 3.1 Data Source

The mutation data was manually curated from published research papers. Each entry represents a missense mutation that was experimentally tested (usually by electrophysiology) and classified as LOF or GOF.

### 3.2 Raw Data Format

The raw Excel file (`nachr_db_cleaned.xlsx`) has these columns:

| Column | Example | Description |
|--------|---------|-------------|
| OID | 1 | Row identifier |
| nAChR subunit | CHRNA7 | Which gene the mutation is in |
| Modification type | Substitution | Type of mutation (we only use substitutions) |
| AA position | 97 | Position in the protein sequence |
| Initial AA | E | Wildtype (original) amino acid |
| New AA | A | Mutant (replacement) amino acid |
| Effect | LOF | Experimentally determined effect |
| Measuring Technique | Electrophysiology | How the effect was measured |
| Pathology | CMS | Associated disease (if known) |
| Reference(PMID) | 33740418 | PubMed ID of the source paper |

### 3.3 Data Cleaning Steps

The raw database has 413 entries. We filter and clean as follows:

1. **Keep only substitutions:** Drop deletions (15), stop codons (10), frameshifts (4) — we need a WT→MT amino acid pair for our features
2. **Drop ambiguous labels:** Remove "LOF/GOF" entries (10) where the effect was unclear
3. **Drop no-net-effect:** Remove "No net effect" entries (~24) because there are too few for a third class (only ~5% of data)
4. **Fix whitespace:** Strip trailing spaces from "GOF " etc.
5. **Validate amino acids:** Ensure all WT and MT amino acids are one of the 20 standard single-letter codes

**Final dataset: 351 substitution mutations (218 LOF, 133 GOF)**

### 3.4 Mutation Distribution by Subunit

The data is heavily skewed — muscle-type subunits (especially CHRNA1 and CHRNE) dominate because Congenital Myasthenic Syndrome is the most-studied nAChR disease.

| Subunit | Count | % of total |
|---------|-------|------------|
| CHRNA1 | 104 | 29.6% |
| CHRNE | 65 | 18.5% |
| CHRNA7 | 40 | 11.4% |
| CHRND | 23 | 6.6% |
| CHRNA4 | 21 | 6.0% |
| CHRNA6 | 18 | 5.1% |
| CHRNB1 | 18 | 5.1% |
| CHRNB2 | 17 | 4.8% |
| CHRNB4 | 15 | 4.3% |
| CHRNA2 | 9 | 2.6% |
| CHRNA3 | 7 | 2.0% |
| CHRNA5 | 5 | 1.4% |
| CHRNB3 | 4 | 1.1% |
| CHRNG | 3 | 0.9% |
| CHRNA9 | 2 | 0.6% |

### 3.5 FASTA Sequence Files

#### What is a FASTA file?

A **FASTA file** is a simple text format for storing protein (or DNA) sequences. It looks like this:

```
>NP_000069.1 neuronal acetylcholine receptor subunit alpha-7 [Homo sapiens]
MRCSLFLVNLF LPAGSGSWLHH PDQKLI PSDL...
KRLKFMKLGIWTYDDKDKFITSAGDHIRLA...
```

The first line (starting with `>`) is the **header** — it contains the accession number, protein name, and organism. Everything after is the amino acid sequence as single-letter codes (A, C, D, E, F, ...).

#### Why do we need FASTA files?

We use them for three things:

1. **Validation:** When the database says "CHRNA7 position 97 is Glutamic acid (E)", we check the FASTA sequence to confirm that position 97 really is E. This catches data entry errors.

2. **Structural alignment:** PDB structures often have different residue numbering than the UniProt/RefSeq sequence (missing loops, extra purification tags, etc.). We align the FASTA sequence to the PDB chain sequence to map "UniProt position 97" → "PDB residue 102" (or whatever it maps to). Without the FASTA, we wouldn't know which 3D atom coordinates correspond to our mutation.

3. **Future MSA conservation analysis:** We'll align sequences across species to measure evolutionary conservation at each position (see Section 3.6).

#### Why do we use the canonical isoform only?

Many human genes produce multiple **isoforms** — slightly different versions of the same protein created by alternative splicing (different exons get included or excluded). For example, CHRNA7 has several isoforms on NCBI:

```
NP_000069.1    (isoform 1 — canonical, 502 amino acids)
NP_001177408.1 (isoform 2 — shorter, missing exon 4)
XP_011519408.1 (isoform X1 — predicted, slightly different)
```

We use **only the canonical isoform (isoform 1)** because:

- **The mutation database uses canonical numbering.** When a paper says "E97A in CHRNA7," they mean position 97 in the canonical sequence. If we used a different isoform, position 97 might be a completely different amino acid (or not exist at all, if an exon is missing).
- **PDB structures correspond to the canonical isoform.** The experimental structures were solved using the full-length canonical protein. Aligning a non-canonical isoform to the PDB would produce wrong position mappings.
- **Consistency.** Everyone in the field uses the canonical isoform as the reference. The NCBI RefSeq accession (e.g., `NP_000069.1`) is the standard identifier.
- **Non-canonical isoforms are often tissue-specific** or exist in low abundance. The mutations we study were characterized in systems expressing the canonical isoform.

The canonical isoform for each subunit is stored in `config.py` as the `REFSEQ_IDS` dictionary, and the actual FASTA files are in `protein_scequences/`.

### 3.6 Multiple Sequence Alignment (MSA) — What It Is and Why It Matters

#### What is an MSA?

A **Multiple Sequence Alignment (MSA)** takes the same protein from multiple species and lines them up so that equivalent positions are in the same column:

```
Human_CHRNA7:   M R C S L F L V N L F L P A G S G S W L H H P D Q K L I P S D L ...
Mouse_Chrna7:   M R C S L F L V N L F L P A G S G S W L H H P D Q K L I P S D L ...
Rat_Chrna7:     M R C S L F L V N L F L P A G S G S W L H H P D Q K L I P S D L ...
Zebrafish_chrna7: M R C S L - - V N L F L P A G S G N W L H H P D Q K - I P S D L ...
                                ^ ^                     ^               ^
                              gaps              different AA       gap = deletion
```

Positions where all species have the **same** amino acid are called **conserved** — evolution has preserved them because they're functionally important. If a position has been the same for 400 million years of evolution (since zebrafish diverged from humans), mutating it is probably damaging.

#### How conservation helps prediction

- A mutation at a **highly conserved** position (all species have the same AA) is more likely to be LOF — evolution "tested" alternatives and rejected them.
- A mutation at a **variable** position (different species have different AAs) is less likely to be damaging — the protein tolerates variation here.

We plan to compute a **conservation score** at each position from an MSA of nAChR sequences across species (human, mouse, rat, chicken, zebrafish, etc.) and add it as a feature. This is a future improvement (see Section 12, Priority 2).

### 3.7 Sequence Matching for Cross-Species Data (Future Work)

Currently our dataset contains only **human** nAChR mutations. A future goal is to include **mouse** nAChR mutation data to increase our training set size (more data = better models).

The challenge: human and mouse nAChR proteins are very similar (~85-95% identical) but not identical. A mutation at "mouse Chrna7 position 100" might correspond to "human CHRNA7 position 103" because of small insertions or deletions between the species.

**Sequence matching** (pairwise alignment) solves this. We align the mouse sequence to the human canonical sequence, and the alignment tells us which positions correspond:

```
Human:  M R C S L F L V N L F L P ...    (position 97 = E)
Mouse:  M R C S L F L V N L F L P ...    (position 97 = E)
        | | | | | | | | | | | | |
        All match → mouse pos 97 = human pos 97
```

For positions where the sequences differ or have gaps, the alignment adjusts the numbering. This way, mouse mutations can be mapped to equivalent human positions, allowing us to:
1. Combine mouse + human data for training
2. Train on one species and test on the other (species transfer experiment, like ENaC did)

This is planned for later — first we need to collect the mouse mutation data.

---

## 4. Feature Engineering (The Core of This Project)

Feature engineering is the process of converting raw mutation information ("CHRNA7 E97A") into a vector of numbers that a machine learning model can learn from. This is the most important part of the project — the choice of features determines how well the model can distinguish LOF from GOF.

### 4.0 The Big Picture

Each mutation is represented as **44 numbers** (without structural features) or **50 numbers** (with structural features). These numbers capture different aspects of what the mutation "looks like" to the protein:

```
Input:  CHRNA7 E97A (LOF)

                    ┌─ WT properties (8 numbers): What is Glutamic acid like?
                    │  MT properties (8 numbers): What is Alanine like?
  Physicochemical ──┤  Differences  (8 numbers): How different are they?
  (24 features)     │
                    └─ e.g., E is charged, A is neutral → diff_charge = +0.5

                    ┌─ BLOSUM62 score: Is E→A common in evolution? (no, score = -1)
  Substitution ─────┤  BLOSUM62 normalized: Same, scaled to [0,1]
  (3 features)      └─ Grantham distance: How physically different? (107/215 = moderate)

                    ┌─ Position normalized: Where in the protein? (97/1314 = 0.074, near N-terminus)
  Positional ───────┤
  (17 features)     └─ Subunit one-hot: Which gene? [0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0] = CHRNA7

                    ┌─ RSA: Is position 97 buried or exposed?
                    │  B-factor: Is it rigid or flexible?
  Structural ───────┤  DSSP: Is it in a helix, sheet, or loop?
  (6 features)      │  C-beta density: How tightly packed is this region?
                    └─ (currently imputed — needs PDB files)

Output: [0.458, 0.914, 0.426, ..., 1.0, 0.0, ..., 0.0]  ← 44 numbers total
```

### 4.1 Physicochemical Features (24 features)

**File:** `vep_nachr/features/physicochemical.py`  
**From ENaC:** `VEP-Enac/vep/features/engineered.py` — taken with no changes.

#### What are amino acid properties?

There are 20 standard amino acids, each with different physical and chemical properties. These properties determine how the amino acid behaves in a protein. The **AAIndex** database (https://www.genome.jp/aaindex/) is a collection of hundreds of published scales that quantify these properties.

We use 8 carefully chosen scales:

#### Property 1: Hydrophobicity (EISD840101)
- **What:** How much does this amino acid avoid water?
- **Range:** Leucine (L) is the most hydrophobic (1.0), Arginine (R) is the least (0.0)
- **Why it matters:** Proteins fold so that hydrophobic residues are buried inside (away from water) and hydrophilic residues are on the surface. If you replace a buried hydrophobic residue with a hydrophilic one, the protein may misfold → LOF.
- **Example:** L→R (hydrophobic→hydrophilic) at a buried position = almost certainly damaging

#### Property 2: Polarity (GRAR740102)
- **What:** How unevenly distributed is the electron cloud?
- **Range:** Aspartate (D) is most polar (1.0), Leucine (L) is least (0.0)
- **Why it matters:** Polar residues form hydrogen bonds. Losing or gaining polarity can break critical interactions.

#### Property 3: Volume (KRIW790103)
- **What:** How physically large is the side chain?
- **Range:** Tryptophan (W) is largest (1.0), Glycine (G) is smallest (0.0)
- **Why it matters:** Putting a large amino acid where a small one was (or vice versa) causes steric clashes or creates cavities. In tightly packed regions, even small size changes can be disruptive.
- **Example:** G→W (tiny→huge) in the channel pore would block ion flow.

#### Property 4: Molecular Weight (FASG760101)
- **What:** The mass of the amino acid in Daltons.
- **Why it matters:** Correlated with volume but captures additional chemistry (heavier atoms, more complex side chains).

#### Property 5: Charge at pH 7 (KLEP840101)
- **What:** Net electric charge at physiological pH.
- **Key values:** D, E = negative (0.0). K, R = positive (1.0). All others = neutral (0.5).
- **Why it matters:** Charged residues form salt bridges (electrostatic bonds). Losing a charge can break these critical structural contacts. Also, the nAChR ion pore has rings of charged residues that control ion selectivity — mutations here are often GOF or LOF.
- **Example:** E→A at the pore lining removes a negative charge → changes ion conductance.

#### Property 6: Isoelectric Point (ZIMJ680104)
- **What:** The pH at which the amino acid has zero net charge.
- **Why it matters:** Gives a more nuanced view of charge behavior across different pH environments.

#### Property 7: Aromaticity (binary)
- **What:** Does the side chain contain an aromatic ring?
- **Values:** F (Phenylalanine), W (Tryptophan), Y (Tyrosine), H (Histidine) = 1. All others = 0.
- **Why it matters:** Aromatic residues participate in pi-stacking interactions and are important for ligand binding. In nAChRs, aromatic residues in the binding pocket form the "aromatic box" that coordinates acetylcholine.

#### Property 8: Secondary Structure Preference (CHOP780201)
- **What:** The tendency of this amino acid to be found in alpha-helices (Chou & Fasman scale).
- **Range:** Glutamate (E) has the highest helix preference (1.0), Glycine (G) the lowest (0.0).
- **Why it matters:** The TM domains of nAChRs are alpha-helices. Introducing a helix-breaking residue (like Proline) into TM2 would disrupt the pore structure → likely LOF.

#### How we compute them

For each mutation, we compute **three sets** of values:

1. **WT properties** (8 values): Look up the wildtype amino acid in the table
2. **MT properties** (8 values): Look up the mutant amino acid in the table
3. **Differences** (8 values): MT - WT for each property

The differences are the most informative — they tell the model "how much did this property change?" A large diff_charge means the mutation dramatically altered the electrostatics. A large diff_hydrophobicity means a polar↔hydrophobic swap.

All values are **min-max normalized to [0, 1]** across the 20 standard amino acids, so different scales are comparable.

#### Example: CHRNA7 E97A

```
Property                    WT (E)   MT (A)   Diff (A-E)
─────────────────────────   ──────   ──────   ─────────
hydrophobicity               0.458    0.807    +0.349  ← A is more hydrophobic
polarity                     0.914    0.395    -0.519  ← A is much less polar
volume                       0.426    0.189    -0.237  ← A is smaller
molecular_weight             0.558    0.109    -0.449  ← A is lighter
charge                       0.000    0.500    +0.500  ← E is negative, A is neutral!
isoelectric_point            0.056    0.404    +0.348
aromaticity                  0.000    0.000    +0.000  ← neither has a ring
secondary_structure_pref     1.000    0.904    -0.096  ← both like helices
```

The model sees that this mutation removes a negative charge (diff_charge = +0.5) and shifts from polar to hydrophobic (diff_polarity = -0.52). These are significant changes that could disrupt a salt bridge → LOF.

### 4.2 Substitution Scores (3 features)

**File:** `vep_nachr/features/substitution.py`  
**From ENaC:** BLOSUM62 part taken from ENaC. Grantham distance is NEW (ENaC did not have it).

#### BLOSUM62 — Evolutionary Substitution Score

**What is it?** BLOSUM62 (BLOcks SUbstitution Matrix) is a 20×20 matrix that scores every possible amino acid pair. It was built by analyzing real protein evolution — comparing related proteins and counting how often each substitution actually occurs.

- **High score** (e.g., D→E = +2): These amino acids are frequently swapped in evolution. They have similar properties, so the swap is tolerated. "Conservative substitution."
- **Low score** (e.g., C→W = -4): This swap is almost never seen in evolution. The amino acids are very different. "Radical substitution."
- **Diagonal** (e.g., W→W = +15): No change. Maximum score.

**Why it's useful:** If evolution has "tested" this substitution and found it acceptable (high BLOSUM score), the mutation is probably less damaging. If evolution has avoided it (low BLOSUM score), it's probably disruptive.

We store:
- `blosum62_raw`: The raw integer score (-4 to 11 for substitutions)
- `blosum62_normalized`: Scaled to [0, 1] for the model

#### Grantham Distance — Physicochemical Distance (NEW)

**What is it?** A composite distance metric published by Grantham in 1974. It combines three physicochemical properties (composition, polarity, molecular volume) into a single number that represents how "different" two amino acids are.

- **Range:** 0 (identical amino acids) to 215 (Cysteine → Tryptophan, the most different pair)
- **Low distance** (e.g., I→L = 5): Very similar amino acids. Conservative change.
- **High distance** (e.g., C→W = 215): Extremely different. Radical change.

**Why we added it (ENaC didn't have it):**
BLOSUM62 is based on **statistical observation** — how often does this substitution happen in evolution? Grantham is based on **physicochemical theory** — how different are these amino acids in terms of measurable properties? They capture complementary information. A substitution might be rare in evolution (low BLOSUM) not because it's damaging, but because the codon change requires multiple nucleotide mutations. Grantham cuts through this by looking only at the physical difference.

We normalize to [0, 1] by dividing by 215 (the max).

#### Example: E→A
```
BLOSUM62 raw:    -1   (moderately penalized — not a common evolutionary swap)
BLOSUM62 norm:   0.20 (on a [0,1] scale)
Grantham raw:    107  (moderate physicochemical distance)
Grantham norm:   0.50 (on a [0,1] scale)
```

### 4.3 Positional and Subunit Features (17 features)

**File:** `vep_nachr/features/encoder.py`  
**From ENaC:** Adapted (16 subunits instead of 3, no species encoding).

#### Normalized Position (1 feature)

The amino acid position of the mutation, divided by the maximum position in the dataset.

```
position_normalized = mutation_position / 1314
```

(1314 is the longest sequence position in our dataset)

**Why it matters:** nAChR subunits have a defined domain structure:
- Positions ~1-20: Signal peptide (cleaved off, no mutations here)
- Positions ~20-210: Extracellular domain (ACh binding site)
- Positions ~210-430: Transmembrane domains (TM1-TM4, the pore)
- Positions ~310-450: Intracellular loop (between TM3-TM4)
- Positions ~430-500+: C-terminal extracellular region

So the position tells the model roughly which domain the mutation is in. Mutations in the transmembrane domain (especially TM2, which lines the pore) tend to have more severe effects.

#### Subunit One-Hot Encoding (16 features)

Each of the 16 nAChR genes gets a binary column. For a mutation in CHRNA7, the CHRNA7 column is 1 and all others are 0.

**Why it matters:** Different subunits have different roles, expression patterns, and mutation sensitivities:
- CHRNA1 mutations are mostly LOF (muscle CMS)
- CHRNA4 mutations are often GOF (epilepsy)
- The model needs to know which subunit to adjust its predictions

**ENaC had 3 subunit columns** (alpha, beta, gamma). We have 16 because nAChR has many more genes.

### 4.4 Structural Features (6 features — NOW IMPLEMENTED)

**File:** `vep_nachr/features/structural.py`  
**From ENaC:** `VEP-Enac/vep/features/noah_features/structural_features.py` — adapted for multi-PDB support.

**Status: WORKING** — B-factor, DSSP, and C-beta density are extracted from PDB structures. RSA requires the `mkdssp` binary (not available on Windows) and is currently imputed as 1.0.

#### How It Works (Step by Step)

1. **Load the CIF structure file** using BioPython's MMCIFParser
2. **Align the UniProt sequence to the PDB chain** using BLOSUM62 global alignment (because PDB residue numbering doesn't always match UniProt numbering — there can be missing loops, extra residues, etc.)
3. **For each mutation**, look up which PDB residue corresponds to the UniProt position
4. **Extract features** from that residue's 3D coordinates

#### PDB Chain Assignments (determined by sequence alignment)

| PDB ID | Chain | Subunit | Mutations Covered |
|--------|-------|---------|-------------------|
| 7QKO | A | CHRNA1 | 104 |
| 7QKO | B | CHRNB1 | 18 |
| 7QKO | C | CHRND | 23 |
| 7QKO | D | CHRNE | 65 |
| 7QKO | E | CHRNG | 3 |
| 7EKI | A | CHRNA7 | 40 |
| 6CNJ | A | CHRNA4 | 21 |
| 6CNJ | B | CHRNB2 | 17 |
| 6PV7 | A | CHRNA3 | 7 |
| 6PV7 | B | CHRNB4 | 15 |

**Coverage: 267/351 mutations (76%) mapped to PDB, 84/351 imputed**

The 84 imputed mutations are from subunits without experimental structures (CHRNA2, CHRNA5, CHRNA6, CHRNA9, CHRNB3) or positions that fall outside the resolved structure.

#### Feature Extraction Results

| Feature | Min | Max | Mean | Status |
|---------|-----|-----|------|--------|
| rsa | 1.0 | 1.0 | 1.0 | Imputed (needs mkdssp binary) |
| bfactor | 0.0 | 211.0 | 71.6 | **Working** — real data from PDB |
| dssp_helix | 0 | 1 | 0.28 | **Working** — from mmCIF annotations |
| dssp_sheet | 0 | 1 | 0.17 | **Working** — from mmCIF annotations |
| dssp_coil | 0 | 1 | 0.56 | **Working** — from mmCIF annotations |
| cbeta_density | 0 | 26 | 11.7 | **Working** — from 3D coordinates |

#### What are PDB/CIF Files?

A **PDB/CIF file** contains the 3D coordinates (x, y, z) of every atom in a protein, determined by experiments like X-ray crystallography or cryo-electron microscopy. CIF (Crystallographic Information File) is the modern format replacing the older PDB format.

#### Feature: RSA (Relative Solvent Accessibility)

- **What:** What fraction of this residue's surface is exposed to water (solvent)?
- **Range:** 0.0 (completely buried inside the protein) to 1.0 (fully exposed on the surface)
- **How it's computed:** The DSSP algorithm calculates the solvent-accessible surface area (ASA) for each residue. We divide by the maximum possible ASA for that amino acid type:
  ```
  RSA = ASA / MaxASA[amino_acid_type]
  ```
- **Currently imputed as 1.0** because the `mkdssp` binary is not available on Windows. To get real RSA values, install mkdssp (Linux/Mac) or use a pre-computed DSSP file.
- **Why it matters:** Buried residues (RSA < 0.2) are in the protein core. Mutations here are more likely to be damaging.

#### Feature: B-factor (Temperature Factor)

- **What:** How much does this residue move/vibrate in the structure?
- **Unit:** Angstroms squared (Å²)
- **How it's computed:** Average B-factor of all heavy (non-hydrogen) atoms in the residue:
  ```
  B_factor = mean([atom.bfactor for atom in residue if atom.element != 'H'])
  ```
- **Fallback chain:** If the residue is missing, try adjacent residues, then chain median
- **Why it matters:** Low B-factor = rigid, structurally important. High B-factor = flexible, often in loops.

#### Feature: DSSP Secondary Structure (3 binary features)

- **What:** Is this residue in an alpha-helix, beta-sheet, or coil/loop?
- **How it's computed:** Parsed from mmCIF secondary structure annotations:
  - Helix annotations (HELX_*) → `dssp_helix = 1`
  - Sheet annotations (SHEET) → `dssp_sheet = 1`
  - Everything else → `dssp_coil = 1`
- **Why it matters:** In nAChRs, the transmembrane helices (TM1-TM4) are critical. A mutation that breaks a helix is likely damaging.

#### Feature: C-beta Density

- **What:** How many amino acids are packed around this position within a 10 Angstrom radius?
- **How it's computed:**
  1. Collect all C-beta atom coordinates in the chain (C-alpha for Glycine)
  2. Build a KDTree (spatial search structure) for fast neighbor queries
  3. Count neighbors within 10Å, subtract 1 (to exclude self)
- **Why it matters:** High density = tightly packed core. Mutations here are more disruptive.

#### What is AlphaFold?

**AlphaFold** is an AI system by Google DeepMind that predicts a protein's 3D structure from its amino acid sequence alone. For subunits without experimental structures (CHRNA2, CHRNA5, CHRNA6, CHRNA9, CHRNB3), AlphaFold structures can be downloaded from https://alphafold.ebi.ac.uk/ by searching the UniProt ID.

### 4.5 Feature Summary Table

| Index | Feature Name | Group | Range | What It Captures |
|-------|-------------|-------|-------|------------------|
| 0-7 | wt_{property} | Physicochemical | [0, 1] | Properties of the original amino acid |
| 8-15 | mt_{property} | Physicochemical | [0, 1] | Properties of the replacement amino acid |
| 16-23 | diff_{property} | Physicochemical | [-1, 1] | How much each property changed |
| 24 | blosum62_raw | Substitution | [-4, 11] | Evolutionary likelihood of this swap |
| 25 | blosum62_normalized | Substitution | [0, 1] | Same, scaled |
| 26 | grantham_normalized | Substitution | [0, 1] | Physicochemical distance between AAs |
| 27 | position_normalized | Positional | [0, 1] | Where in the protein (N-term to C-term) |
| 28-43 | subunit_{name} | Positional | {0, 1} | Which nAChR gene (one-hot) |
| 44 | rsa | Structural | [0, 1] | Solvent exposure (buried vs surface) |
| 45 | bfactor | Structural | [0, ∞) | Atomic mobility (rigid vs flexible) |
| 46 | dssp_helix | Structural | {0, 1} | In an alpha-helix? |
| 47 | dssp_sheet | Structural | {0, 1} | In a beta-sheet? |
| 48 | dssp_coil | Structural | {0, 1} | In a loop/coil? |
| 49 | cbeta_density | Structural | [0, ~30] | Packing density (how crowded) |

**Total: 44 features** (working) **→ 50 features** (when structural is implemented)

---

## 5. Machine Learning Pipeline

### 5.1 The Classification Task

Given a 44-dimensional feature vector representing a mutation, predict whether it causes **LOF (0)** or **GOF (1)**.

This is a **binary classification** problem with **imbalanced classes** (62% LOF vs 38% GOF).

### 5.2 Why We Use Multiple Models

No single ML algorithm is best for all problems (this is called the "No Free Lunch" theorem). Different models make different assumptions:

| Model | How It Works (Simple Explanation) | Strengths | Weaknesses |
|-------|----------------------------------|-----------|------------|
| **Logistic Regression** | Draws a straight line (hyperplane) to separate LOF and GOF in feature space | Interpretable, fast, good baseline | Can't capture non-linear patterns |
| **SVM (RBF)** | Finds the best separating boundary, but can curve it using the "kernel trick" | Good with small data, handles non-linearity | Sensitive to feature scaling, slow to tune |
| **Random Forest** | Builds 100+ decision trees on random subsets, takes majority vote | Handles non-linearity, robust, gives feature importance | Can overfit with too many trees |
| **LightGBM / XGBoost** | Builds trees sequentially, each one correcting the errors of the previous | Often best for tabular data, fast | Can overfit, harder to interpret |
| **KNN** | Looks at the K most similar mutations in the training set, takes majority vote | Simple, no assumptions | Slow at prediction time, sensitive to irrelevant features |
| **MLP** | A small neural network with one hidden layer | Can learn any function | Needs lots of data (we don't have enough) |
| **Gaussian NB** | Assumes each feature independently follows a bell curve per class | Very fast, works with small data | Strong independence assumption rarely holds |

### 5.3 Handling Class Imbalance

We have 218 LOF vs 133 GOF (62% vs 38%). If the model just predicts LOF for everything, it gets 62% accuracy "for free." To prevent this:

- **`class_weight='balanced'`**: Most models support this. It tells the model to penalize misclassifying GOF samples more heavily, proportional to how rare they are. Internally, it multiplies the loss for GOF samples by (218/133 ≈ 1.64).
- **Stratified splitting**: When dividing data into train/test folds, we ensure each fold has the same LOF/GOF ratio as the full dataset.

### 5.4 Evaluation Metrics

#### F1 Score (Primary Metric)

F1 is the harmonic mean of precision and recall for the GOF class:

```
Precision = True GOF predictions / All GOF predictions
            "When the model says GOF, how often is it right?"

Recall    = True GOF predictions / All actual GOF mutations
            "Of all real GOF mutations, how many does the model catch?"

F1        = 2 × (Precision × Recall) / (Precision + Recall)
            Balances both — penalizes if either is low
```

F1 = 0.654 means the model is moderately good at finding GOF mutations but still misses some.

#### Accuracy (Secondary Metric)

```
Accuracy = Correct predictions / Total predictions
```

Simple but misleading with imbalanced data. A "predict all LOF" baseline gets 62%.

#### Confusion Matrix

```
                    Predicted LOF    Predicted GOF
Actual LOF              TN               FP
Actual GOF              FN               TP

TN = True Negative:  Correctly predicted LOF
FP = False Positive: Predicted GOF but was actually LOF
FN = False Negative: Predicted LOF but was actually GOF
TP = True Positive:  Correctly predicted GOF
```

### 5.5 Cross-Validation: How We Split the Data

We NEVER train and evaluate on the same data. That would be like letting a student see the exam answers before taking the test — the score would be meaninglessly high.

#### Quick Mode (`--quick`): Simple 5-Fold CV × 5 Seeds

```
351 mutations
  ├── Fold 1: [70 test] [281 train]  ← train model, test on held-out 70
  ├── Fold 2: [70 test] [281 train]  ← different 70 held out
  ├── Fold 3: [70 test] [281 train]
  ├── Fold 4: [70 test] [281 train]
  └── Fold 5: [71 test] [280 train]
                                      ← every mutation gets tested exactly once

This is repeated 5 times with different random shuffles (seeds).
Total: 5 folds × 5 seeds = 25 evaluations → report mean ± std
```

**Split ratio: 80% train / 20% test** (rotating so all data is tested)

No hyperparameter tuning. Uses default model settings.

#### Full Mode: Nested CV (5 Outer × 5 Inner × 5 Seeds)

```
351 mutations
  ├── Outer Fold 1: [70 TEST] [281 for training]
  │     └── Inner optimization on the 281:
  │           ├── Inner Fold 1: [56 valid] [225 train] ← try HP set #1, score = 0.62
  │           ├── Inner Fold 2: [56 valid] [225 train] ← try HP set #1, score = 0.65
  │           ├── ... (50 Optuna trials × 5 inner folds)
  │           └── Best HPs found → train on all 281 → predict on 70 TEST
  ├── Outer Fold 2: [70 TEST] [281 for training]
  │     └── (same inner optimization)
  └── ...
```

**Split ratio: 64% train / 16% validation (inner) / 20% test (outer)**

This is the gold standard. The test set is NEVER seen during hyperparameter tuning, so the reported score is an honest estimate of real-world performance.

### 5.6 Hyperparameter Optimization (Optuna)

Every ML model has settings (hyperparameters) that affect how it learns:
- Random Forest: how many trees? how deep can each tree be?
- SVM: how much regularization? how flexible is the boundary?
- LightGBM: learning rate? number of leaves?

**Optuna** is a library that automatically searches for the best hyperparameters. It uses a smart search strategy called TPE (Tree-structured Parzen Estimator) that learns from previous trials to focus on promising regions of the search space.

We run 50 Optuna trials per fold. Each trial:
1. Suggests a set of hyperparameters
2. Trains the model with those HPs on the inner training set
3. Evaluates on the inner validation set
4. Reports the F1 score back to Optuna

After 50 trials, we take the best-performing HP set and use it for the outer fold evaluation.

### 5.7 Feature Scaling

Some models (Logistic Regression, SVM, KNN, MLP) are sensitive to feature scales. If one feature ranges from 0-1 and another from 0-1000, the model will be dominated by the larger one.

We use **RobustScaler** (from sklearn) which scales each feature by its interquartile range. This is more robust to outliers than standard scaling (mean/std).

Tree-based models (Random Forest, LightGBM, XGBoost) are **not** affected by feature scales — they make split decisions that are scale-invariant.

---

## 6. Comparison with VEP-ENaC (The Reference Project)

This project is adapted from VEP-ENaC, a Variant Effect Predictor for the Epithelial Sodium Channel (ENaC) built by a senior colleague.

### 6.1 The Proteins

| Aspect | ENaC | nAChR |
|--------|------|-------|
| **Full name** | Epithelial Sodium Channel | Nicotinic Acetylcholine Receptor |
| **Ion channel type** | Constitutively open Na+ channel | Ligand-gated cation channel |
| **Oligomeric state** | Trimer (3 subunits) | Pentamer (5 subunits) |
| **Subunit genes** | 3 (α, β, γ) | 17 (CHRNA1-10, CHRNB1-4, CHRND, CHRNE, CHRNG) |
| **Superfamily** | DEG/ENaC | Cys-loop receptors |
| **Gating** | Always open, regulated by proteases | Opens when acetylcholine binds |

Both are ion channels, so the feature engineering approach (physicochemical properties, structural context, substitution scores) is applicable to both. The biology is different enough that the exact weights the model learns will differ, but the feature types are the same.

### 6.2 Code Reuse Summary

| Component | Our File | ENaC Source | Changes Made |
|-----------|----------|-------------|-------------|
| Physicochemical features | `features/physicochemical.py` | `features/engineered.py` | **None** — amino acid properties are universal |
| BLOSUM62 scores | `features/substitution.py` | `features/engineered.py` | **Minor** — extracted into own module |
| Grantham distance | `features/substitution.py` | N/A | **New** — ENaC didn't have this |
| Structural features | `features/structural.py` | `features/noah_features/structural_features.py` | **Skeleton** — needs adaptation for multi-PDB |
| Feature encoder | `features/encoder.py` | `features/engineered.py` | **Adapted** — 16 subunits instead of 3, no species |
| Model registry | `models/registry.py` | `models/registry.py` | **Adapted** — binary scoring, removed CatBoost |
| Cross-validation | `training/cross_validation.py` | `training/cross_validation.py` | **Simplified** — removed species transfer, binary F1 |
| Data loader | `data/loader.py` | `data/` (multiple files) | **Rewritten** — Excel format, different columns |
| Config | `config.py` | `config.py` | **Rewritten** — nAChR subunits, binary labels |

### 6.3 What We Did NOT Take from ENaC

| ENaC Component | What It Was | Why We Skipped It |
|----------------|-------------|------------------|
| Noah's Original Features | A second feature pipeline with different AAIndex properties | Redundant — we consolidated into one cleaner pipeline |
| Data-Driven Encoders | Ordinal, one-hot, full-sequence encoding alternatives | Engineered features performed best in ENaC |
| Species Transfer Experiment | Train on human vs mouse vs both | We only have human data (mouse coming later) |
| Ablation Studies | Remove one feature group at a time to measure impact | Will add once baseline is solid |
| CatBoost model | Yandex gradient boosting library | Sklearn compatibility issues; XGBoost/LightGBM sufficient |
| Paper/figure scripts | LaTeX, matplotlib publication figures | Not needed yet |

### 6.4 What We Added That ENaC Didn't Have

| Addition | Description |
|----------|-------------|
| Grantham distance | Physicochemical distance score (complementary to BLOSUM62) |
| 16-way subunit encoding | Much more subunit resolution (ENaC had only 3) |

---

## 7. Results

### 7.1 Baseline (Quick Mode, No HP Tuning, No Structural Features)

351 samples (218 LOF, 133 GOF), 44 features, 5-fold CV x 5 seeds:

| Model | F1 (GOF) | +/- Std | Accuracy | +/- Std | Notes |
|-------|----------|---------|----------|---------|-------|
| **XGBoost** | **0.655** | 0.070 | **0.744** | 0.051 | Best F1 |
| **LightGBM** | **0.654** | 0.068 | 0.737 | 0.051 | Very close to XGBoost |
| KNN | 0.654 | 0.090 | 0.746 | 0.046 | Best accuracy, high variance |
| Logistic Regression | 0.648 | 0.067 | 0.706 | 0.046 | Solid linear baseline |
| Random Forest | 0.635 | 0.079 | 0.737 | 0.057 | Expected to improve with HP tuning |
| SVM (RBF) | 0.600 | 0.069 | 0.664 | 0.051 | Needs HP tuning |
| Gaussian NB | 0.588 | 0.028 | 0.502 | 0.066 | Bad accuracy (near random) |
| MLP | 0.422 | 0.126 | 0.663 | 0.058 | Unstable, not enough data |

### 7.2 With Structural Features (Quick Mode, No HP Tuning)

351 samples (218 LOF, 133 GOF), **50 features**, 5-fold CV x 5 seeds:

| Model | F1 (w/ struct) | F1 (w/o struct) | Change | Accuracy |
|-------|----------------|-----------------|--------|----------|
| **XGBoost** | **0.666** | 0.655 | +0.011 | 0.755 |
| **LightGBM** | **0.666** | 0.654 | +0.012 | 0.748 |
| Logistic Regression | 0.653 | 0.648 | +0.005 | 0.717 |
| Random Forest | 0.627 | 0.635 | -0.008 | 0.743 |

Structural features provide a **modest improvement** (+1-2% F1 for gradient boosting models). Note: RSA is still imputed (mkdssp unavailable on Windows), so once that's fixed, the improvement could be larger.

### 7.3 Comparison with ENaC

ENaC's best result was F1 ~ 0.538 (macro, 3-class). Our best is F1 ~ 0.666 (binary). This isn't an apples-to-apples comparison because:
1. Binary classification (2 classes) is inherently easier than 3-class
2. F1_binary and F1_macro are computed differently
3. Different proteins, different data distributions

But it's encouraging that the same feature engineering approach works for a different protein family.

---

## 8. Evaluation Metrics — What They Mean and What's a "Good" Score

### 8.1 Why Not Just Use Accuracy?

Our dataset has 218 LOF and 133 GOF (62% vs 38%). If a model just predicted **LOF for every single mutation**, it would get:

```
Accuracy = 218 / 351 = 62.1%
```

That's a terrible model — it learned nothing — but 62% accuracy sounds okay. This is why accuracy is misleading for imbalanced data. We need metrics that care about **how well the model identifies GOF** (the minority class).

### 8.2 F1 Score — Our Primary Metric

F1 score combines two things:

**Precision:** "When the model says GOF, how often is it actually GOF?"
```
Precision = TP / (TP + FP)

If the model predicts 35 mutations as GOF:
  - 28 of them are actually GOF (True Positives = 28)
  - 7 of them are actually LOF (False Positives = 7, the model was wrong)

Precision = 28 / (28 + 7) = 28/35 = 0.80
→ "80% of the model's GOF predictions are correct"
```

**Recall:** "Of all the real GOF mutations, how many does the model catch?"
```
Recall = TP / (TP + FN)

There are 40 actual GOF mutations in the test set:
  - The model correctly identifies 28 of them (True Positives = 28)
  - The model misses 12 of them, calling them LOF (False Negatives = 12)

Recall = 28 / (28 + 12) = 28/40 = 0.70
→ "The model catches 70% of the GOF mutations"
```

**F1 is the harmonic mean of precision and recall:**
```
F1 = 2 × (Precision × Recall) / (Precision + Recall)
   = 2 × (0.80 × 0.70) / (0.80 + 0.70)
   = 2 × 0.56 / 1.50
   = 0.747
```

Why harmonic mean and not regular average? Because the harmonic mean **penalizes extremes**. If precision is 1.0 but recall is 0.0 (model predicts GOF very rarely but is always right when it does), the regular average would be 0.50, but F1 = 0.0. The model is useless — it catches no GOF mutations — and F1 reflects that.

### 8.3 What's a "Good" F1 Score?

For **variant effect prediction on small, curated datasets** like ours:

| F1 Score | Interpretation |
|----------|---------------|
| 0.50 | Barely above random — model is struggling |
| 0.60 | Below average — some signal but lots of errors |
| 0.65-0.70 | **Typical for this type of task** with ~350 samples and engineered features |
| 0.70-0.80 | Good — model is learning meaningful patterns |
| 0.80+ | Very good — either great features or possible overfitting |
| 0.90+ | Suspicious — check for data leakage or overfitting |

Our current best is **F1 = 0.666** — in the typical range for this data size and feature set. This means the model is capturing real signal (not just guessing) but there's room for improvement.

**Random baseline for our data:**
A model that randomly guesses GOF with the true class frequency (38%) would get:
```
Expected Precision = 0.38 (since 38% of data is GOF)
Expected Recall    = 0.38 (random guess catches 38% of GOFs)
Expected F1        = 0.38
```

Our F1 of 0.666 is substantially above this, confirming the features contain real predictive signal.

### 8.4 The ± Standard Deviation

When we report F1 = 0.666 ± 0.070, the ± 0.070 is the **standard deviation across 25 evaluations** (5 folds × 5 seeds). It tells you how much the score varies depending on which mutations happen to be in the test set.

- **Low std (± 0.03):** Consistent — the model performs similarly regardless of which mutations are tested. Trustworthy.
- **High std (± 0.12):** Unstable — performance depends heavily on which mutations are in the test set. The model might be memorizing specific examples rather than learning general patterns. Less trustworthy.

Our MLP has std = 0.126 (very unstable) while LightGBM has std = 0.068 (more stable). This makes sense — neural networks need more data to learn stably, and 351 mutations isn't enough.

### 8.5 Confusion Matrix — The Full Picture

The confusion matrix shows all four outcomes at once:

```
                    Predicted LOF    Predicted GOF
                    ─────────────    ─────────────
Actual LOF  │         TN = 53           FP = 7
            │     "Correctly said      "Said GOF but
            │      it's LOF"            was LOF"
            │
Actual GOF  │         FN = 12           TP = 28
            │     "Missed this GOF,    "Correctly found
            │      called it LOF"       this GOF"
```

From this you can see exactly WHERE the model makes mistakes:
- FP = 7: The model is too aggressive — it calls 7 LOF mutations GOF (could lead to wrong clinical advice)
- FN = 12: The model is too conservative — it misses 12 GOF mutations (could miss important drug targets)

In clinical settings, which type of error is worse depends on the application. For drug discovery, missing a GOF (FN) might mean missing a therapeutic target. For diagnosis, calling something GOF when it's LOF (FP) might lead to wrong treatment.

---

## 9. How Features Are Converted to Numbers (The Complete Pipeline)

This section explains exactly what happens to a mutation from raw text to the numbers the model sees. No hand-waving — every step is shown.

### 9.1 The Full Pipeline for One Mutation

Let's trace **CHRNA7 E97A** through the entire pipeline:

```
Raw input:   "CHRNA7"  "E"  "A"  97  "LOF"
                │        │    │    │     │
                ▼        ▼    ▼    ▼     ▼
Step 1: Physicochemical lookup (24 numbers)
Step 2: Substitution score lookup (3 numbers)
Step 3: Position normalization (1 number)
Step 4: Subunit one-hot encoding (16 numbers)
Step 5: Structural feature extraction (6 numbers)
Step 6: Concatenate all → [50 numbers]
Step 7: Scale with RobustScaler → [50 scaled numbers]
Step 8: Feed to model → prediction (0 or 1)
```

### 9.2 Step 1: Physicochemical Lookup (24 numbers)

We have a **pre-built lookup table** with 20 rows (one per amino acid) and 8 columns (one per property). All values are min-max normalized to [0, 1].

The lookup:
```
Table["E"] → [0.458, 0.914, 0.426, 0.558, 0.000, 0.056, 0.000, 1.000]
Table["A"] → [0.807, 0.395, 0.189, 0.109, 0.500, 0.404, 0.000, 0.904]
```

Then we compute the difference:
```
Diff = Table["A"] - Table["E"]
     → [+0.349, -0.519, -0.237, -0.449, +0.500, +0.348, 0.000, -0.096]
```

Concatenate all three: `[WT(8) | MT(8) | Diff(8)]` = **24 numbers**

These are NOT rounded to 0 or 1. They are continuous decimal values between 0 and 1 (for WT/MT) or between -1 and +1 (for differences). The only exception is aromaticity, which is binary (0 or 1) because an amino acid either has an aromatic ring or it doesn't.

### 9.3 Step 2: Substitution Score Lookup (3 numbers)

**BLOSUM62:** Look up the pair (E, A) in a 20x20 matrix:
```
BLOSUM62["E"]["A"] = -1   (raw integer score)
```

Normalize to [0, 1]:
```
blosum62_normalized = (-1 - (-4)) / (11 - (-4)) = 3/15 = 0.200
```
(where -4 is the minimum and 11 is the maximum score in the BLOSUM62 matrix for the 20 standard amino acids)

**Grantham distance:** Look up the pair (E, A) in a distance table:
```
Grantham["E"]["A"] = 107   (raw distance)
grantham_normalized = 107 / 215 = 0.498
```
(215 is the maximum possible Grantham distance, between Cysteine and Tryptophan)

Result: `[-1, 0.200, 0.498]` = **3 numbers**

### 9.4 Step 3: Position Normalization (1 number)

Simple division:
```
position_normalized = 97 / 1314 = 0.0738
```
(1314 is the maximum mutation position in our dataset, determined during fit)

Result: `[0.0738]` = **1 number** (continuous, between 0 and 1)

### 9.5 Step 4: Subunit One-Hot Encoding (16 numbers)

Create a binary vector with 16 slots, one per subunit. Set the slot for this mutation's subunit to 1, all others to 0:

```
[CHRNA1, CHRNA2, CHRNA3, CHRNA4, CHRNA5, CHRNA6, CHRNA7, CHRNA9, CHRNA10,
 CHRNB1, CHRNB2, CHRNB3, CHRNB4, CHRND, CHRNE, CHRNG]

CHRNA7 → [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
```

These ARE strictly 0 or 1. This is the only part of our features that uses pure binary encoding.

Result: `[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]` = **16 numbers**

### 9.6 Step 5: Structural Feature Extraction (6 numbers)

For CHRNA7 position 97, we look up PDB 7EKI chain A:

1. **Align** the canonical FASTA sequence to the PDB chain sequence using BLOSUM62 pairwise alignment → find that UniProt position 97 maps to PDB residue X
2. **Read B-factor** from the PDB atom coordinates: average B-factor of heavy atoms at that residue → e.g., `72.5`
3. **Read secondary structure** from mmCIF annotations → e.g., helix → `[1, 0, 0]`
4. **Compute C-beta density** by counting C-beta atoms within 10 Angstroms using a KDTree → e.g., `14`
5. **RSA** → currently imputed as `1.0` (mkdssp not available)

Result: `[1.0, 72.5, 1.0, 0.0, 0.0, 14.0]` = **6 numbers** (continuous, not rounded)

### 9.7 Step 6: Concatenate Everything

All features are joined into a single flat array (a 1D vector):

```
[0.458, 0.914, ..., 0.200, 0.498, 0.074, 0, 0, 0, 0, 0, 0, 1, ..., 1.0, 72.5, ...]
 ├─── 24 physico ───┤├─ 3 subst ─┤├ pos ┤├──── 16 subunit one-hot ────┤├─ 6 struct ──┤

Total: 50 numbers per mutation
```

This is stored as a **NumPy array** with shape `(1, 50)` — one row of 50 floating-point numbers. For the whole dataset, the feature matrix has shape `(351, 50)` — 351 mutations, each represented by 50 numbers.

```python
X = np.array([
    [0.458, 0.914, ..., 14.0],   # mutation 1: CHRNA7 E97A
    [0.302, 0.667, ..., 8.0],    # mutation 2: CHRNA1 R209H
    ...                           # ... 349 more rows
])
# X.shape = (351, 50)
```

### 9.8 Step 7: Feature Scaling (RobustScaler)

Before feeding features to scale-sensitive models (Logistic Regression, SVM, KNN, MLP), we scale each feature column independently.

**RobustScaler** uses the median and interquartile range (IQR) instead of mean and standard deviation:

```
For each feature column j:
    median_j = median of all 351 values in column j
    IQR_j    = Q75_j - Q25_j   (75th percentile minus 25th percentile)

    scaled_value = (original_value - median_j) / IQR_j
```

Example for the bfactor column:
```
All 351 bfactor values: [72.5, 45.2, 98.3, 0.0, 0.0, 55.1, ...]
median = 65.0
Q25 = 40.0,  Q75 = 90.0,  IQR = 50.0

CHRNA7 E97A bfactor = 72.5
scaled = (72.5 - 65.0) / 50.0 = 0.15
```

After scaling, features are centered around 0 with similar spreads. This matters because models like SVM use distance calculations — if bfactor ranges from 0-200 while hydrophobicity ranges 0-1, the model would be dominated by bfactor without scaling.

**Tree-based models** (Random Forest, XGBoost, LightGBM) skip this step — they make split decisions like "is bfactor > 50?" which are unaffected by scaling.

### 9.9 Step 8: What the Model Actually Sees

After all transformations, each mutation is a row of 50 numbers. The label (LOF=0, GOF=1) is a separate integer. The model sees:

```
Features (X):                                              Label (y):
[0.15, -0.82, 0.33, ..., 0.04, -0.10, 0.28, ...]         0 (LOF)
[0.52, 0.14, -0.45, ..., 1.20, 0.55, -0.33, ...]         1 (GOF)
[0.08, -0.90, 0.67, ..., -0.50, -0.22, 0.15, ...]        0 (LOF)
...
```

The model's job: find patterns in these 50 numbers that distinguish LOF rows from GOF rows. For example, XGBoost might learn rules like:
- "If diff_charge > 0.3 AND position_normalized is between 0.15-0.30 AND subunit_CHRNA4 = 1, predict GOF"
- "If diff_hydrophobicity < -0.4 AND dssp_helix = 1 AND bfactor < 50, predict LOF"

It learns these rules automatically from the training data — we don't tell it what patterns to look for.

### 9.10 Summary: Are Features Rounded? Are They Matrices?

| Question | Answer |
|----------|--------|
| Are features rounded to 0 or 1? | **No** — most features are continuous decimals (0.458, -0.519, 72.5, etc.). Only subunit one-hot (16 features) and DSSP (3 features) are binary 0/1. |
| Are they stored as matrices? | **Yes** — the feature matrix X has shape (351, 50). Each row is one mutation, each column is one feature. Stored as a NumPy 2D array of float64 values. |
| What data type? | All values are 64-bit floating point numbers (float64), even the binary ones (stored as 0.0 and 1.0). |
| Is anything discretized? | The raw BLOSUM62 score is an integer (-4 to 11), but it's stored as float64 alongside everything else. Nothing is binned or bucketed. |
| How is the label stored? | As a 1D NumPy array of integers: `y = [0, 1, 0, 0, 1, ...]` where 0=LOF, 1=GOF. Shape: (351,). |

---

## 10. How to Run the Code

### 10.1 Setup

```bash
cd "VEP Nachr"
pip install -r requirements.txt
```

### 10.2 Quick Test (Default Hyperparameters, No Optuna)

```bash
# Without structural features (fast, no PDB needed):
python scripts/run_experiment.py --quick --no-structural

# With structural features (requires CIF files in data/raw/structure_files/):
python scripts/run_experiment.py --quick
```

### 10.3 Full Experiment (With Optuna HP Optimization)

```bash
python scripts/run_experiment.py
```

This runs nested cross-validation with 50 Optuna trials per fold. Much slower but gives more accurate performance estimates.

### 10.4 Single Model

```bash
python scripts/run_experiment.py --model random_forest
```

### 10.5 All Models (Core + Extended)

```bash
python scripts/run_experiment.py --all-models --quick
```

### 10.6 Flags

| Flag | What it does |
|------|-------------|
| `--quick` | Use default HPs, skip Optuna optimization |
| `--no-structural` | Exclude structural features (use when no PDB files available) |
| `--model NAME` | Run only one model (e.g., `random_forest`, `lightgbm`) |
| `--all-models` | Run all 9 models instead of just the 4 core ones |
| `--n-trials N` | Number of Optuna trials per fold (default: 50) |

---

## 11. Project File Map

```
VEP Nachr/
│
├── NOTES.ipynb                              # This documentation file
├── requirements.txt                         # Python dependencies (pip install -r requirements.txt)
│
├── scripts/
│   └── run_experiment.py                    # Main entry point for running experiments
│
├── vep_nachr/                               # The Python package (all source code)
│   ├── __init__.py                          # Package marker, version info
│   ├── config.py                            # All settings: paths, subunits, PDB mappings, labels
│   │
│   ├── data/
│   │   ├── __init__.py
│   │   └── loader.py                        # Load Excel DB, clean data, load FASTA sequences
│   │
│   ├── features/
│   │   ├── __init__.py
│   │   ├── physicochemical.py               # 24 AAIndex features (from ENaC, unchanged)
│   │   ├── substitution.py                  # BLOSUM62 (from ENaC) + Grantham distance (new)
│   │   ├── structural.py                    # PDB-based features (WORKING for B-factor, DSSP, C-beta)
│   │   └── encoder.py                       # Combines all features into sklearn transformer
│   │
│   ├── models/
│   │   ├── __init__.py
│   │   └── registry.py                      # 9 ML models + Optuna HP search spaces
│   │
│   └── training/
│       ├── __init__.py
│       └── cross_validation.py              # Nested CV, simple CV, result saving
│
├── data/
│   ├── raw/
│   │   └── structure_files/                 # CIF files: 7QKO.cif, 7EKI.cif, 6CNJ.cif, 6PV7.cif
│   └── processed/                           # For cleaned/intermediate data files
│
└── results/                                 # Experiment JSON results saved here
```

---

## 12. Next Steps / Roadmap

### Priority 1: Improve Current Pipeline
1. **Get RSA working:** Install mkdssp on Linux/Mac, or generate DSSP files externally and load them. RSA is potentially the most informative structural feature.
2. **Download AlphaFold structures** for the 5 uncovered subunits (CHRNA2, CHRNA5, CHRNA6, CHRNA9, CHRNB3) to cover the remaining 38 mutations.
3. **Run full Optuna HP optimization** (`python scripts/run_experiment.py --all-models`) and compare with baseline.
4. **Clean duplicate mutations** in the database (same mutation reported by different papers).

### Priority 2: Add New Features
5. **Domain annotations:** Add features for which structural domain the mutation is in (ECD, TM1-4, ICD). Mutations in TM2 (pore lining) are particularly important.
6. **Distance to binding site:** How far is the mutation from the ACh binding pocket? Closer = more likely to affect function.
7. **Conservation scores:** Align nAChR sequences across species (human, mouse, rat, zebrafish, etc.) and compute how conserved each position is.

### Priority 3: Advanced Features
8. **ESM embeddings:** Use protein language models (ESM-2 from Meta) to generate per-residue embeddings. These are learned representations from millions of protein sequences and often outperform hand-crafted features.
9. **Collect mouse nAChR mutation data** and implement species transfer experiment (like ENaC did).

### Priority 4: Analysis
10. **Feature ablation studies:** Remove one feature group at a time to measure which features contribute most.
11. **SHAP feature importance:** Visualize which features the model relies on for individual predictions.

---

## 13. Appendix A: Amino Acid Transfer Statistics

This section analyzes the mutation "traffic" in our dataset — which amino acids get mutated most, what they get mutated to, and whether certain substitution pairs are strongly associated with LOF or GOF.

### 13.1 Which Amino Acids Get Mutated Most? (Wildtype Frequency)

These are the amino acids that appear as the **original** (wildtype) residue in our mutations. Higher count = this amino acid is mutated more often in our dataset.

| WT AA | Name | Count | LOF | GOF | % LOF | Interpretation |
|-------|------|-------|-----|-----|-------|----------------|
| **V** | Valine | 50 | 25 | 25 | 50% | Most mutated. Perfect LOF/GOF split — context-dependent |
| **R** | Arginine | 33 | 23 | 10 | 70% | Charged residue, mutations usually damaging |
| **L** | Leucine | 32 | 13 | 19 | 41% | Hydrophobic, mutations lean GOF |
| **D** | Aspartate | 25 | 22 | 3 | 88% | Negatively charged, almost always LOF when mutated |
| **S** | Serine | 25 | 11 | 14 | 44% | Small polar, slight GOF lean |
| **E** | Glutamate | 24 | 24 | 0 | **100%** | **Every single E mutation is LOF** |
| **T** | Threonine | 24 | 15 | 9 | 62% | Slightly LOF-leaning |
| **P** | Proline | 22 | 15 | 7 | 68% | Helix breaker, LOF-leaning |
| **N** | Asparagine | 20 | 15 | 5 | 75% | Polar, usually LOF |
| **F** | Phenylalanine | 19 | 13 | 6 | 68% | Aromatic, LOF-leaning |
| **Y** | Tyrosine | 16 | 12 | 4 | 75% | Aromatic, LOF-leaning |
| **C** | Cysteine | 13 | 7 | 6 | 54% | Split — depends on disulfide bond involvement |
| **I** | Isoleucine | 11 | 4 | 7 | 36% | Hydrophobic, GOF-leaning |
| **G** | Glycine | 9 | 3 | 6 | 33% | Smallest AA, GOF-leaning when mutated |
| **M** | Methionine | 6 | 2 | 4 | 33% | GOF-leaning |
| **Q** | Glutamine | 6 | 5 | 1 | 83% | LOF-leaning |
| **K** | Lysine | 6 | 4 | 2 | 67% | Charged, LOF-leaning |
| **A** | Alanine | 5 | 2 | 3 | 40% | Small, neutral |
| **W** | Tryptophan | 3 | 3 | 0 | 100% | Largest AA, always LOF when mutated |

**Key insight:** Glutamate (E) mutations are 100% LOF (24/24). This makes biological sense — E is negatively charged and often forms salt bridges or lines the ion pore. Losing glutamate always breaks something. Tryptophan (W) is also always LOF — it's the largest amino acid and plays critical structural roles.

### 13.2 Which Amino Acids Are the Most Common Replacements? (Mutant Frequency)

| MT AA | Name | Count | LOF | GOF | % LOF | Interpretation |
|-------|------|-------|-----|-----|-------|----------------|
| **A** | Alanine | 51 | 37 | 14 | 73% | Most common replacement, usually LOF |
| **L** | Leucine | 42 | 30 | 12 | 71% | Second most common, usually LOF |
| **F** | Phenylalanine | 28 | 10 | 18 | 36% | **GOF-leaning replacement** |
| **S** | Serine | 21 | 11 | 10 | 52% | Neutral |
| **K** | Lysine | 21 | 17 | 4 | 81% | Introducing positive charge = LOF |
| **T** | Threonine | 20 | 8 | 12 | 40% | GOF-leaning replacement |
| **C** | Cysteine | 20 | 16 | 4 | 80% | LOF — can form unwanted disulfide bonds |
| **M** | Methionine | 12 | 2 | 10 | **17%** | **Strongest GOF-leaning replacement** |

**Key insight:** Mutating TO methionine (M) is 83% GOF — the strongest GOF signal of any replacement. Mutating TO alanine (A) or cysteine (C) is strongly LOF.

### 13.3 Net Gain/Loss Per Amino Acid

Which amino acids are "gaining" mutations (more mutations TO them) vs "losing" them (more mutations FROM them)?

| AA | Mutated FROM | Mutated TO | Net | Direction |
|----|-------------|-----------|-----|-----------|
| A | 5 | 51 | **+46** | Massive gainer — most common destination |
| K | 6 | 21 | +15 | Gains — charge introduction |
| L | 32 | 42 | +10 | Moderate gainer |
| F | 19 | 28 | +9 | Moderate gainer |
| V | 50 | 8 | **-42** | Massive loser — Valine gets mutated away |
| R | 33 | 15 | -18 | Loses — Arginine breaks easily |
| D | 25 | 8 | -17 | Loses — Aspartate is vulnerable |
| E | 24 | 12 | -12 | Loses — Glutamate is vulnerable |

**Biological interpretation:** Valine (V) is the most "fragile" position — it gets mutated away 50 times but is only the replacement 8 times. This is likely because V is extremely common in the transmembrane helices of nAChRs (TM1-TM4 are hydrophobic), so there are simply more V residues available to mutate. Alanine (A) is the most common destination, which makes sense — A is the smallest hydrophobic residue, often used in mutagenesis studies as a "deletion" that removes the side chain while keeping the backbone.

### 13.4 Top 25 Substitution Pairs

| Pair | Count | LOF | GOF | % LOF | Pattern |
|------|-------|-----|-----|-------|---------|
| V->L | 11 | 8 | 3 | 73% | Conservative hydrophobic swap, still mostly LOF |
| D->N | 11 | 10 | 1 | 91% | Charge removal (neg -> neutral), strongly LOF |
| F->L | 11 | 9 | 2 | 82% | Aromatic -> aliphatic, LOF |
| V->A | 11 | 5 | 6 | 45% | Size reduction, context-dependent |
| E->A | 10 | 10 | 0 | **100%** | Charge removal, **always LOF** |
| R->C | 9 | 6 | 3 | 67% | Charge removal + disulfide risk |
| P->L | 9 | 6 | 3 | 67% | Helix flexibility change |
| **L->F** | **8** | **1** | **7** | **12%** | **Aliphatic -> aromatic, strongly GOF** |
| E->K | 7 | 7 | 0 | **100%** | Charge reversal (neg -> pos), **always LOF** |
| L->P | 7 | 5 | 2 | 71% | Helix breaker, LOF |
| R->W | 7 | 6 | 1 | 86% | Charge -> aromatic, LOF |
| **G->S** | **6** | **1** | **5** | **17%** | **Size increase, strongly GOF** |
| **V->M** | **6** | **1** | **5** | **17%** | **Hydrophobic swap, strongly GOF** |
| D->E | 5 | 4 | 1 | 80% | Conservative (both neg), still LOF |
| V->I | 5 | 5 | 0 | 100% | Conservative swap, always LOF |
| **V->T** | **5** | **1** | **4** | **20%** | **Hydrophobic -> polar, GOF** |
| **L->T** | **5** | **0** | **5** | **0%** | **Always GOF** |
| **T->I** | **5** | **1** | **4** | **20%** | **Polar -> hydrophobic, GOF** |
| **V->F** | **4** | **0** | **4** | **0%** | **Always GOF** |

### 13.5 LOF-Dominant vs GOF-Dominant Pairs

**Pairs that are almost always LOF** (>= 3 occurrences, >= 80% LOF):

| Pair | Count | % LOF | Why |
|------|-------|-------|-----|
| E->A | 10 | 100% | Removes negative charge entirely |
| E->K | 7 | 100% | Reverses charge from negative to positive |
| E->R | 4 | 100% | Charge reversal |
| D->A | 4 | 100% | Removes negative charge |
| D->N | 11 | 91% | Removes charge, keeps similar size |
| V->I | 5 | 100% | Very conservative, still always LOF |
| T->A | 4 | 100% | Removes hydroxyl group |
| T->C | 3 | 100% | Introduces cysteine (disulfide risk) |
| R->W | 7 | 86% | Removes positive charge |
| F->L | 11 | 82% | Removes aromatic ring |
| S->P | 3 | 100% | Introduces helix-breaking proline |
| C->S | 3 | 100% | Breaks disulfide bond |

**Pairs that are almost always GOF** (>= 3 occurrences, >= 80% GOF):

| Pair | Count | % GOF | Why |
|------|-------|-------|-----|
| L->T | 5 | 100% | Hydrophobic -> polar in TM domain? |
| V->F | 4 | 100% | Introduces aromatic bulk |
| S->Y | 3 | 100% | Introduces aromatic ring + size increase |
| L->F | 8 | 88% | Aliphatic -> aromatic (bigger, pi-stacking) |
| G->S | 6 | 83% | Adds side chain where there was none |
| V->M | 6 | 83% | Subtle hydrophobic change, GOF |
| T->I | 5 | 80% | Polar -> hydrophobic |
| V->T | 5 | 80% | Hydrophobic -> polar |

### 13.6 What Does This Tell Us?

1. **Charge changes are almost always LOF.** Removing or reversing a charge (E->A, E->K, D->N, R->W) is one of the strongest LOF signals. The model should weigh `diff_charge` heavily.

2. **Aromatic introductions tend toward GOF.** L->F, V->F, S->Y all introduce aromatic rings and lean GOF. This could be because aromatic residues in the transmembrane domain can alter pore properties or ligand interactions in ways that increase channel activity.

3. **Hydrophobic -> polar swaps in TM domains are often GOF.** V->T, L->T are GOF-dominant. Introducing polarity into the hydrophobic transmembrane helices may alter ion conductance or gating without destroying the channel.

4. **Conservative substitutions can still be 100% LOF.** V->I (very conservative — both branched hydrophobics) is always LOF. This shows that even small changes at critical positions are enough to break function. Position matters as much as the chemical change.

5. **These statistics could become features.** A "historical LOF rate for this substitution pair" feature could be informative, though with small counts per pair there's overfitting risk.

---

## 14. Appendix B: Claude Code Skills

This section explains what **Claude Code skills** are, how they work, and how we set one up in this project. This is a reference for anyone using Claude Code (the AI coding assistant) in this workspace.

### 14.1 What Is a Skill?

A **skill** is a reusable instruction file that tells Claude Code how to handle a specific type of task. Instead of typing a long prompt every time (e.g., "interview me about every aspect of my design, ask one question at a time, provide recommendations..."), you just type `/grill-me` and Claude loads the full instructions automatically.

Think of it like a macro or a template — you define the behavior once, and invoke it with a short command.

### 14.2 How Skills Work

A skill is just a **markdown file** named `SKILL.md` placed in a specific folder. It has two parts:

**1. YAML frontmatter** (metadata between `---` markers):
```yaml
---
name: grill-me
description: Interview the user relentlessly about a plan or design...
---
```

**2. Markdown body** (the actual instructions Claude follows):
```markdown
Interview me relentlessly about every aspect of this plan...
Ask the questions one at a time.
If a question can be answered by exploring the codebase, explore the codebase instead.
```

When you type `/grill-me`, Claude reads this file and follows the instructions as if you had typed them yourself.

### 14.3 Where Skills Live

Skills are stored as `SKILL.md` files in a `.claude/skills/` directory:

```
your-project/
├── .claude/
│   ├── settings.local.json      # Claude Code permissions
│   └── skills/                   # Skills directory
│       └── grill-me.md          # A skill file
├── src/
└── ...
```

There are two scopes:

| Location | Scope | Who sees it |
|----------|-------|-------------|
| `~/.claude/skills/skill-name/SKILL.md` | **Personal** | You, across all projects |
| `your-project/.claude/skills/skill-name.md` | **Project** | Anyone working on this project |

Our `grill-me` skill is at the **project level** so it's available to anyone who clones this repo.

### 14.4 How to Invoke a Skill

- **Type `/` in Claude Code** to see a list of all available skills
- **Type `/skill-name`** to invoke a specific skill (e.g., `/grill-me`)
- **With arguments:** `/skill-name your argument here` (e.g., `/grill-me our feature engineering approach`)

Claude can also **auto-invoke** skills based on context — if you mention "grill me about this design," Claude may automatically load the skill because its description matches.

### 14.5 The `grill-me` Skill (Installed in This Project)

**Location:** `.claude/skills/grill-me.md`

**What it does:** When invoked, Claude will systematically interrogate you about every aspect of a plan, design, or decision. It walks through each branch of the decision tree, resolving dependencies one by one, and provides its own recommended answer for each question.

**When to use it:**
- Before implementing a major feature — stress-test your design
- When planning architecture decisions — find blind spots
- Before a meeting or presentation — prepare for tough questions
- When you're unsure about an approach — let Claude challenge your assumptions

**Example usage:**
```
You: /grill-me our plan to add conservation scores as features

Claude: Let's systematically work through your conservation feature plan.

Question 1: Which species will you include in the MSA?
My recommendation: Start with human, mouse, rat, chicken, zebrafish, and 
Xenopus. These cover ~400M years of evolution with well-annotated genomes.

[waits for your answer, then moves to next question...]
```

### 14.6 Skill Frontmatter Reference

These are the key fields you can put in the YAML frontmatter:

| Field | What it does | Example |
|-------|-------------|---------|
| `name` | Display name | `grill-me` |
| `description` | What the skill does (helps Claude decide when to auto-load) | `"Interview the user..."` |
| `disable-model-invocation` | If `true`, only YOU can invoke it (Claude won't auto-load) | `true` |
| `allowed-tools` | Tools Claude can use without asking permission | `Bash(git *) Read(*)` |
| `argument-hint` | Hint shown during autocomplete | `[topic]` |
| `context` | Set to `fork` to run in an isolated subagent | `fork` |

### 14.7 How to Add More Skills

**From a GitHub repo (like mattpocock/skills):**

1. Find the skill you want (e.g., `teach`, `tdd`, `caveman`)
2. Copy the `SKILL.md` content
3. Create a new file at `.claude/skills/skill-name.md` in your project
4. Paste the content

**Write your own:**

1. Create `.claude/skills/your-skill-name.md`
2. Add YAML frontmatter with `name` and `description`
3. Write the instructions in markdown
4. Invoke with `/your-skill-name`

**Example — a custom skill for this project:**
```yaml
---
name: check-features
description: Validate that all feature engineering steps produce correct output shapes and ranges
---

Run the feature pipeline on a small sample (5 mutations) and verify:
1. Physicochemical features: shape (5, 24), values in [0, 1]
2. Substitution features: shape (5, 3), BLOSUM62 raw in [-4, 11]
3. Structural features: shape (5, 6), DSSP values are binary
4. Final encoded matrix: shape (5, 50)

Report any anomalies.
```

### 14.8 Available Skills from mattpocock/skills Repository

Some useful skills from `github.com/mattpocock/skills` that could be added:

| Skill | Category | What it does |
|-------|----------|-------------|
| `grill-me` | Productivity | Systematic design interrogation (installed) |
| `teach` | Productivity | Claude explains concepts at your level |
| `tdd` | Engineering | Test-driven development workflow |
| `diagnose` | Engineering | Structured debugging workflow |
| `zoom-out` | Engineering | Request broader context analysis |
| `caveman` | Productivity | Compressed communication mode (fewer words) |
| `prototype` | Engineering | Build experimental implementations quickly |
| `handoff` | Productivity | Transfer work context between sessions |